In [2]:
import pandas as pd
from pathlib import Path

# Ruta directa
ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
df = pd.read_parquet(ROOT / "data" / "06_evaluacion" / "meta_test.parquet")

# Ver todas las titulaciones que contienen "Edificación" o "Arquitectura"
mask = df["titulacion"].str.contains("Edificaci|Arquitectura", case=False, na=False)
print(df[mask]["titulacion"].value_counts())

# Ver todas las titulaciones con coma
mask2 = df["titulacion"].str.contains(",", na=False)
print("\nTitulaciones con coma:")
print(df[mask2]["titulacion"].value_counts())

titulacion
Grado en Arquitectura Técnica                94
Grado en Ingeniería de la Edificación        41
Grado en Arquitectura Técnica (Plan 2020)     8
Name: count, dtype: int64

Titulaciones con coma:
titulacion
Doble Grado en Administración y Dirección de Empresas y Derecho,     22
Name: count, dtype: int64


In [3]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
df = pd.read_parquet(ROOT / "data" / "06_evaluacion" / "meta_test.parquet")

col_anio = "anio_cohorte" if "anio_cohorte" in df.columns else "curso_aca_ini"

tabla = (
    df.groupby("titulacion")[col_anio]
    .agg(inicio="min", fin="max", n_alumnos="count")
    .reset_index()
    .sort_values(["inicio", "titulacion"])
)

# Clasificar tipo
def tipo_carrera(nombre):
    if nombre.startswith("Doble Grado"):
        return "Doble Grado"
    elif nombre.startswith("Grado"):
        return "Grado"
    else:
        return "Otro"

tabla["tipo"] = tabla["titulacion"].apply(tipo_carrera)
print(tabla[["tipo", "titulacion", "inicio", "fin", "n_alumnos"]].to_string(index=False))

       tipo                                                         titulacion  inicio  fin  n_alumnos
      Grado                                      Grado en Arquitectura Técnica    2009 2019         94
      Grado                                  Grado en Comunicación Audiovisual    2009 2020        234
      Grado                                         Grado en Estudios Ingleses    2009 2020        161
      Grado                              Grado en Ingeniería de la Edificación    2009 2011         41
      Grado                                                Grado en Periodismo    2009 2020        228
      Grado                          Grado en Publicidad y Relaciones Públicas    2009 2020        256
      Grado                                                   Grado en Química    2009 2020        177
      Grado                   Grado en Relaciones Laborales y Recursos Humanos    2009 2020        213
      Grado                               Grado en Traducción e Interpret

In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
df = pd.read_parquet(ROOT / "data" / "06_evaluacion" / "meta_test.parquet")

col_anio = "anio_cohorte" if "anio_cohorte" in df.columns else "curso_aca_ini"

tabla = (
    df.groupby("titulacion")[col_anio]
    .agg(inicio="min", fin="max", n_alumnos="count")
    .reset_index()
    .sort_values(["inicio", "titulacion"])
)

def tipo_carrera(nombre):
    if nombre.startswith("Doble Grado"):
        return "Doble Grado"
    elif "(Plan" in nombre:
        return "Plan renovado"
    elif nombre.endswith(","):
        return "Error datos"
    else:
        return "Grado"

def estado(row):
    if row["fin"] < 2018:
        return "Extinguida"
    elif row["fin"] == 2020:
        return "Activa"
    else:
        return "Posible extinción"

tabla["tipo"]   = tabla["titulacion"].apply(tipo_carrera)
tabla["estado"] = tabla.apply(estado, axis=1)
tabla.columns   = ["Titulación", "Inicio", "Fin", "Alumnos test", "Tipo", "Estado"]
tabla = tabla[["Tipo", "Estado", "Titulación", "Inicio", "Fin", "Alumnos test"]]

tabla.to_excel(ROOT / "titulaciones_revision.xlsx", index=False)
print(f"Guardado en: {ROOT / 'titulaciones_revision.xlsx'}")
print(tabla.to_string(index=False))

Guardado en: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\titulaciones_revision.xlsx
         Tipo            Estado                                                         Titulación  Inicio  Fin  Alumnos test
        Grado Posible extinción                                      Grado en Arquitectura Técnica    2009 2019            94
        Grado            Activa                                  Grado en Comunicación Audiovisual    2009 2020           234
        Grado            Activa                                         Grado en Estudios Ingleses    2009 2020           161
        Grado        Extinguida                              Grado en Ingeniería de la Edificación    2009 2011            41
        Grado            Activa                                                Grado en Periodismo    2009 2020           228
        Grado            Activa                          Grado en Publicidad y Relaciones Públicas    2009 2020           256
        Grado        

In [9]:


ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")

# Cargar dataset completo (tiene más historia que meta_test)
df = pd.read_parquet(ROOT / "data" / "03_features" / "df_alumno_limpio.parquet") # ← esta línea


col_anio = "anio_cohorte" if "anio_cohorte" in df.columns else "curso_aca_ini"
col_ab   = "abandono"

# -------------------------------------------------------
# BLOQUE 1 — Humanidades: ¿son la misma carrera?
# -------------------------------------------------------
mask_hu = df["titulacion"].str.contains("Humanidades", case=False, na=False)
hu = (
    df[mask_hu]
    .groupby(["titulacion", col_anio])
    .size()
    .reset_index(name="n_alumnos")
    .sort_values(["titulacion", col_anio])
)
print("=== HUMANIDADES — alumnos por año ===")
print(hu.to_string(index=False))

# -------------------------------------------------------
# BLOQUE 2 — Abandono real por pareja de planes
# -------------------------------------------------------
parejas = [
    ("Grado en Arquitectura Técnica",                                    "Grado en Arquitectura Técnica (Plan 2020)"),
    ("Grado en Criminologia y Seguridad",                                "Grado en Criminologia y Seguridad  (Plan 2020)"),
    ("Grado en Historia y Patrimonio",                                   "Grado en Historia y Patrimonio (Plan 2015)"),
    ("Grado en Ingeniería Agroalimentaria y del Medio Rural",            "Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)"),
    ("Grado en Maestro en Educación Infantil",                           "Grado en Maestro en Educación Infantil (Plan 2018)"),
    ("Grado en Maestro en Educación Primaria",                           "Grado en Maestro en Educación Primaria (Plan 2018)"),
    ("Grado en Medicina",                                                "Grado en Medicina (Plan 2017)"),
    ("Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos","Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos (Plan 2020)"),
    ("Grado en Humanidades: Estudios Interculturales",                   "Grado en Humanidades: Estudios Interculturales"),
]

print("\n=== ABANDONO REAL POR PAREJA ===")
filas = []
for plan_viejo, plan_nuevo in parejas:
    for nombre in [plan_viejo, plan_nuevo]:
        sub = df[df["titulacion"] == nombre]
        if sub.empty:
            continue
        filas.append({
            "Titulación":      nombre,
            "N alumnos":       len(sub),
            "Inicio":          sub[col_anio].min(),
            "Fin":             sub[col_anio].max(),
            "Tasa abandono":   f"{sub[col_ab].mean()*100:.1f}%" if col_ab in sub.columns else "N/D",
        })

df_res = pd.DataFrame(filas)
print(df_res.to_string(index=False))
df_res.to_excel(ROOT / "parejas_planes_abandono.xlsx", index=False)
print(f"\nGuardado en: {ROOT / 'parejas_planes_abandono.xlsx'}")

=== HUMANIDADES — alumnos por año ===
                                    titulacion  curso_aca_ini  n_alumnos
Grado en Humanidades: Estudios Interculturales           2010        174
Grado en Humanidades: Estudios Interculturales           2011        151
Grado en Humanidades: Estudios Interculturales           2012        156
Grado en Humanidades: Estudios Interculturales           2013         85
Grado en Humanidades: Estudios Interculturales           2014        112
Grado en Humanidades: Estudios Interculturales           2015         71
Grado en Humanidades: Estudios Interculturales           2016        111
Grado en Humanidades: Estudios Interculturales           2017         84
Grado en Humanidades: Estudios Interculturales           2018         56
Grado en Humanidades: Estudios Interculturales           2019         52
Grado en Humanidades: Estudios Interculturales           2020         34

=== ABANDONO REAL POR PAREJA ===
                                                    

In [10]:
df = pd.read_parquet(ROOT / "data" / "03_features" / "df_alumno_limpio.parquet")
print(df.columns.tolist())
print(df.shape)

['per_id_ficticio', 'exp_tit_id', 'curso_aca_ini', 'curso_aca', 'curso_aca_fin', 'nota_expediente', 'via_acceso_exp', 'seguro', 'nota_selectividad', 'nota_acceso', 'cred_matriculados', 'cred_superados', 'egresado', 'nuevo', 'media_curso', 'titulacion', 'rama', 'cred_titulacion', 'sexo', 'fecha_nacimiento', 'pais_nombre', 'poblacion', 'provincia', 'vive_fuera', 'nombre_beca', 'tiene_beca', 'nombre_trabajo', 'media_titulacion_curso', 'media_titulacion_alumno', 'forma_de_pago', 'numero_pagos', 'via_acceso', 'orden_preferencia', 'cupo', 'universidad_origen', 'anio_entrada_corregido', 'anio_nacimiento', 'edad_entrada_calc', 'cred_superados_anio', 'indicador_edad_inusual', 'indicador_interrupcion', 'indicador_sin_notas']
(109568, 42)


In [6]:
import os
ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")

# Buscar df_alumno.parquet en todo el proyecto
for ruta in ROOT.rglob("df_alumno*.parquet"):
    print(ruta)

C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed\df_alumno.parquet
C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed\df_alumno_base.parquet
C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\df_alumno_limpio.parquet


In [11]:
df = pd.read_parquet(ROOT / "data" / "automl" / "dataset_final_tfm.parquet")
print(df.columns.tolist())
print(df.shape)

['cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso', 'rama', 'sexo', 'edad_entrada', 'pais_nombre', 'provincia', 'via_acceso', 'orden_preferencia', 'universidad_origen', 'tuvo_beca', 'n_anios_beca', 'situacion_laboral', 'nota_selectividad', 'max_pagos', 'indicador_interrupcion', 'anios_gap', 'anios_sin_beca', 'abandono']
(33621, 20)


In [12]:
df = pd.read_parquet(ROOT / "data" / "06_evaluacion" / "meta_test.parquet")
print(df.columns.tolist())

['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'vive_fuera', 'cupo', 'n_titulaciones', 'flag_cautela']


In [13]:
parejas = [
    ("Grado en Arquitectura Técnica",                                     "Grado en Arquitectura Técnica (Plan 2020)"),
    ("Grado en Ingeniería de la Edificación",                             "Grado en Arquitectura Técnica"),
    ("Grado en Criminologia y Seguridad",                                 "Grado en Criminologia y Seguridad  (Plan 2020)"),
    ("Grado en Historia y Patrimonio",                                    "Grado en Historia y Patrimonio (Plan 2015)"),
    ("Grado en Ingeniería Agroalimentaria y del Medio Rural",             "Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)"),
    ("Grado en Maestro en Educación Infantil",                            "Grado en Maestro en Educación Infantil (Plan 2018)"),
    ("Grado en Maestro en Educación Primaria",                            "Grado en Maestro en Educación Primaria (Plan 2018)"),
    ("Grado en Medicina",                                                 "Grado en Medicina (Plan 2017)"),
    ("Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos","Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos (Plan 2020)"),
]

df = pd.read_parquet(ROOT / "data" / "06_evaluacion" / "meta_test.parquet")

filas = []
for plan_viejo, plan_nuevo in parejas:
    for nombre in [plan_viejo, plan_nuevo]:
        sub = df[df["titulacion"] == nombre]
        if sub.empty:
            continue
        filas.append({
            "Titulación":      nombre,
            "N alumnos":       len(sub),
            "Inicio":          sub["curso_aca_ini"].min(),
            "Fin":             sub["curso_aca_ini"].max(),
            "Tasa abandono":   f"{sub['abandono'].mean()*100:.1f}%",
        })

df_res = pd.DataFrame(filas)
print(df_res.to_string(index=False))
df_res.to_excel(ROOT / "parejas_planes_abandono.xlsx", index=False)
print(f"\nGuardado en: {ROOT / 'parejas_planes_abandono.xlsx'}")

                                                        Titulación  N alumnos  Inicio  Fin Tasa abandono
                                     Grado en Arquitectura Técnica         94    2009 2019         33.0%
                         Grado en Arquitectura Técnica (Plan 2020)          8    2020 2020          0.0%
                             Grado en Ingeniería de la Edificación         41    2009 2011          0.0%
                                     Grado en Arquitectura Técnica         94    2009 2019         33.0%
                                 Grado en Criminologia y Seguridad        190    2010 2019         20.0%
                    Grado en Criminologia y Seguridad  (Plan 2020)         16    2020 2020          0.0%
                                    Grado en Historia y Patrimonio         51    2010 2015         52.9%
                        Grado en Historia y Patrimonio (Plan 2015)         54    2011 2020         20.4%
             Grado en Ingeniería Agroalimentaria y del 